# Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, f1_score

# Importing the Dataset

In [ ]:
data = pd.read_csv("/content/news.csv")
data.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


# Preprocessing Dataset

In [ ]:
data = data.drop(["Unnamed: 0"], axis=1)
data.head()

,title,text,label
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


# Data Encoding

In [ ]:
le = preprocessing.LabelEncoder()
le.fit(data['label'])
y = le.transform(data['label'])

y = to_categorical(y, num_classes=2)
print(y)

[[1. 0.]
 [1. 0.]
 [0. 1.]
 ...
 [1. 0.]
 [0. 1.]
 [0. 1.]]


# Variables Setup

In [ ]:
embedding_dim = 50
max_length = 54
padding_type = 'post'
trunc_type = 'post'
oov_tok = "<OOV>"
training_size = 5000
test_portion = 0.2

# Tokenization (convert sentences into tokens)

In [ ]:
title = []
text = []
labels = []
for x in range(training_size):
    title.append(data['title'][x])
    text.append(data['text'][x])
    labels.append(y[x])

tokenizer1 = Tokenizer()
tokenizer1.fit_on_texts(title) # Fits the tokenizer on the 'title' column to create a vocabulary
word_index1 = tokenizer1.word_index
vocab_size1 = len(word_index1)
sequences1 = tokenizer1.texts_to_sequences(title)
padded1 = pad_sequences(sequences1, padding=padding_type, truncating=trunc_type) # Pads the sequences to ensure they all have the same length

# Splitting Data for Training and Testing

In [ ]:
split = int(test_portion * training_size)
training_sequences1 = padded1[split:training_size]
test_sequences1 = padded1[0:split]
test_labels = labels[0:split]
training_labels = labels[split:training_size]

# Reshaping Data for LSTM

In [ ]:
training_sequences1 = np.array(training_sequences1)
test_sequences1 = np.array(test_sequences1)

# Generating Word Embedding (Words -> Weight Vectors)

In [ ]:
# !wget https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
# !unzip glove.6B.zip

--2026-01-19 12:00:44--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  3.25MB/s    in 4m 11s  

2026-01-19 12:04:55 (3.28 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]

Archive:  glove.6B.zip
  inflating: glove.6B.50d.txt        
  inflating: glove.6B.100d.txt       
  inflating: glove.6B.200d.txt       
  inflating: glove.6B.300d.txt       


In [ ]:
embedding_index = {}
with open('glove.6B.50d.txt', 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coefs

embedding_matrix = np.zeros((vocab_size1 + 1, embedding_dim))

for word, i in word_index1.items():
    if i < vocab_size1:
        embedding_vector = embedding_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

# Model Architecture

In [ ]:
model = keras.Sequential()
model.add(tf.keras.layers.Input(shape=(54,)))
model.add(tf.keras.layers.Embedding(vocab_size1 + 1, embedding_dim, input_length=max_length, weights=[embedding_matrix], trainable=True))
model.add(tf.keras.layers.Dropout(0.2))
model.add(tf.keras.layers.Conv1D(64, 5, activation='relu'))
model.add(tf.keras.layers.MaxPooling1D(pool_size=4))
model.add(tf.keras.layers.LSTM(64))
model.add(tf.keras.layers.Dense(2, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 54, 50)         │       511,650 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 54, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 50, 64)         │        16,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 560,868 (2.14 MB)

 Trainable params: 560,868 (2.14 MB)

 Non-trainable params: 0 (0.00 B)

# Training the Model

In [ ]:
history = model.fit(
    np.array(training_sequences1),
    np.array(training_labels),
    epochs=50,
    validation_data=(np.array(test_sequences1), np.array(test_labels)),
    verbose=2
)

Epoch 1/50
125/125 - 7s - 60ms/step - accuracy: 0.6255 - loss: 0.6304 - val_accuracy: 0.7200 - val_loss: 0.5456
Epoch 2/50
125/125 - 1s - 10ms/step - accuracy: 0.7538 - loss: 0.5124 - val_accuracy: 0.7660 - val_loss: 0.4709
Epoch 3/50
125/125 - 1s - 7ms/step - accuracy: 0.8115 - loss: 0.4097 - val_accuracy: 0.8020 - val_loss: 0.4500
Epoch 4/50
125/125 - 1s - 7ms/step - accuracy: 0.8610 - loss: 0.3220 - val_accuracy: 0.7950 - val_loss: 0.4710
Epoch 5/50
125/125 - 1s - 7ms/step - accuracy: 0.8975 - loss: 0.2495 - val_accuracy: 0.8110 - val_loss: 0.4662
Epoch 6/50
125/125 - 1s - 7ms/step - accuracy: 0.9162 - loss: 0.2061 - val_accuracy: 0.8060 - val_loss: 0.4541
Epoch 7/50
125/125 - 1s - 7ms/step - accuracy: 0.9413 - loss: 0.1480 - val_accuracy: 0.8010 - val_loss: 0.5040
Epoch 8/50
125/125 - 1s - 7ms/step - accuracy: 0.9523 - loss: 0.1206 - val_accuracy: 0.7950 - val_loss: 0.6035
Epoch 9/50
125/125 - 1s - 7ms/step - accuracy: 0.9625 - loss: 0.1000 - val_accuracy: 0.7970 - val_loss: 0.5940

# Prediction

In [ ]:
X = "Karry to go to France in gesture of sympathy"

sequences = tokenizer1.texts_to_sequences([X])
sequences = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)
if model.predict(sequences, verbose=0)[0][0] >= 0.5:
    print("This news is True")
else:
    print("This news is False")

This news is True


In [ ]:
f1_score(np.argmax(test_labels, axis=1), np.argmax(model.predict(np.array(test_sequences1)), axis=1), average='macro')

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


0.8067059998257349

In [ ]:
print(classification_report(np.argmax(test_labels, axis=1), np.argmax(model.predict(np.array(test_sequences1)), axis=1)))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
              precision    recall  f1-score   support

           0       0.83      0.77      0.80       501
           1       0.78      0.85      0.81       499

    accuracy                           0.81      1000
   macro avg       0.81      0.81      0.81      1000
weighted avg       0.81      0.81      0.81      1000

